# Getting started with MiniOps

This notebook introduces the basic MiniOps workflow. A MiniOps operator behaves like a linear map: use `A * x` for the forward operation and `A' * y` for the adjoint.

We will look at padding, convolution, operator diagnostics, composition, and FFT operators. The examples are intentionally small so that this notebook can be used as a quick package check.

In [2]:
using Pkg
Pkg.activate("..")

using MiniOps
using Random
using LinearAlgebra
using Printf
using CairoMakie
using SeisProcessing

  Activating project at `~/MiniOps`


## 1. Padding operator

`pad_op` embeds an array in a larger array by adding samples before and after each dimension. Its adjoint removes the padding. Padding is useful when Fourier-domain operators are combined with interpolation or reconstruction.

In [3]:
n1, n2 = 3, 3
p1, p2 = 2, 2

P = pad_op((n1, n2), (p1, p2), (p1, p2))
X = reshape(1.0:9.0, n1, n2)
XP = P * X

println("size(X)   = ", size(X))
println("size(P*X) = ", size(XP))
XP

size(X)   = (3, 3)
size(P*X) = (7, 7)


7×7 Matrix{Float64}:
 0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  1.0  4.0  7.0  0.0  0.0
 0.0  0.0  2.0  5.0  8.0  0.0  0.0
 0.0  0.0  3.0  6.0  9.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0

## 2. Column-wise convolution and the adjoint

`conv1d_cols_op(h)` applies the same 1-D convolution to every column. The adjoint performs the corresponding correlation operation. The dot-product test verifies

$$
\langle Ax,y\rangle = \langle x,A^*y\rangle.
$$

In [4]:
Random.seed!(0)
h = [1.0, 2.0, -1.0]
A = conv1d_cols_op(h)

nt, ntr = 50, 20
X = randn(nt, ntr)
Y = A * X
Z = randn(size(Y))

ok, err = adjoint_test(A, X, Z)
@show size(X) size(Y) ok err

size(X) = (50, 20)
size(Y) = (52, 20)
ok = true
err = 2.0191521594946724e-15


2.0191521594946724e-15

## 3. Diagnostics and operator composition

MiniOps provides small diagnostic routines that are useful before putting an operator inside an iterative solver. Here we test linearity, adjointness, estimate the operator norm, and compose two scaling operators.

In [5]:
h = randn(14)
B = conv1d_op(h)

println(B.name, " linearity   : ", linearity_test(B, randn(10), randn(10); ntests=3, tol=1e-10))
println(B.name, " adjoint     : ", adjoint_test(B, randn(10), randn(23); tol=1e-10))
println(B.name, " ||B|| approx: ", opnorm_power(B, randn(10); niter=20))

S = scaling_op(3.0)
println(S.name, " self-adjoint: ", is_selfadjoint(S, randn(20); tol=1e-10))

T = scaling_op(0.3) * scaling_op(0.3)
println("composite operator name = ", T.name)

conv1d linearity   : (true, 2.538358523424269e-16)
conv1d adjoint     : (true, 1.4895807184081575e-15)
conv1d ||B|| approx: 6.201713807032337
scaling self-adjoint: (true, 0.0)
composite operator name = scaling*scaling


## 4. Planned and unplanned FFT operators

`fft_op()` is size-agnostic and creates an FFT plan when the operator is applied. `fft_op(x0)` knows the array size in advance and can reuse its plan. Both expose the same forward/adjoint interface.

In [6]:
for F in (fft_op(), fft_op(randn(128)))
    x = randn(128)
    y = randn(128)
    println("
", F.name)
    println("  linearity: ", linearity_test(F, x, randn(128); ntests=3, tol=1e-10))
    println("  adjoint  : ", adjoint_test(F, x, y; tol=1e-10))
    println("  norm     : ", opnorm_power(F, randn(ComplexF64,128); niter=10))
end


fft_op
  linearity: (true, 2.6947763199979797e-16)
  adjoint  : (true, 1.22678819237984e-15)
  norm     : 1.0

fft_planned
  linearity: (true, 2.2788183787840537e-16)
  adjoint  : (true, 2.2959556299699064e-16)
  norm     : 0.9999999999999999


The same constructor works for multidimensional arrays. The next example uses a small 4-D array simply to demonstrate the interface without allocating a large data volume.

In [7]:
x4 = randn(8, 8, 8, 8)
F4 = fft_op(x4)
println("4-D FFT adjoint test: ", adjoint_test(F4, x4, randn(size(x4)); tol=1e-10))
println("operator sizes: m = ", F4.m, ", n = ", F4.n)

4-D FFT adjoint test: (true, 4.2984499110516394e-15)
operator sizes: m = 4096, n = 4096
